# Publishing Feedback

### Setup

In [1]:
# You can set them inline
import os

In [2]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv()

True

### Adding Feedback to a Run

You can add feedback to an existing Run programmatically as long as you know the run_id. Let's grab a run_id from the LangSmith UI and add it here.

In [3]:
run_id = "01a13171-c18b-7b31-85be-af585b362f84"

Now, let's add some continuous feedback

In [4]:
from langsmith import Client

client = Client()

client.create_feedback(
  run_id,
  key="sample-continuous",
  score=7.0,
  comment="This is a sample piece of continuous feedback",
)

C:\Users\reich\AppData\Local\Temp\ipykernel_15428\2728401610.py:5: LangSmithWarning: Creating feedback for a run without session_id is deprecated and will stop working in a future release. See https://docs.langchain.com/langsmith/smithdb-sdk-migration-feedback#feedback-create
  client.create_feedback(


Feedback(id=UUID('01a136a5-d47c-7750-86a8-0bc03874a467'), created_at=datetime.datetime(2026, 10, 13, 19, 3, 19, 933602, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2026, 10, 13, 19, 3, 19, 933606, tzinfo=datetime.timezone.utc), run_id=UUID('01a13171-c18b-7b31-85be-af585b362f84'), trace_id=None, key='sample-continuous', score=7.0, value=None, comment='This is a sample piece of continuous feedback', correction=None, feedback_source=FeedbackSourceBase(type='api', metadata={}, user_id=None, user_name=None), session_id=None, start_time=None, comparative_experiment_id=None, feedback_group_id=None, extra=None)

And now, let's add some categorical feedback too!

In [5]:
from langsmith import Client

client = Client()

client.create_feedback(
  run_id,
  key="sample-categorical",
  value="yes",
  comment="This is a sample piece of categorical feedback",
)

C:\Users\reich\AppData\Local\Temp\ipykernel_15428\1500695314.py:5: LangSmithWarning: Creating feedback for a run without session_id is deprecated and will stop working in a future release. See https://docs.langchain.com/langsmith/smithdb-sdk-migration-feedback#feedback-create
  client.create_feedback(


Feedback(id=UUID('01a136aa-a299-7f51-93a2-79434b9f85f0'), created_at=datetime.datetime(2026, 10, 13, 19, 8, 34, 841700, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2026, 10, 13, 19, 8, 34, 841705, tzinfo=datetime.timezone.utc), run_id=UUID('01a13171-c18b-7b31-85be-af585b362f84'), trace_id=None, key='sample-categorical', score=None, value='yes', comment='This is a sample piece of categorical feedback', correction=None, feedback_source=FeedbackSourceBase(type='api', metadata={}, user_id=None, user_name=None), session_id=None, start_time=None, comparative_experiment_id=None, feedback_group_id=None, extra=None)

### Pre-Generating Run IDs for Feedback

![Generate_run_id](../../images/generate_run_id.png)

Using LangChain, we offer the ability to pre-generate and define run IDs, before your code is invoked and the run ID is generated. With this functionality, you're able to access your run ID before initial generation, which can be useful for actions like sending feedback. The example below demonstrates this.

In [6]:
import uuid

pre_defined_run_id = uuid.uuid4()
pre_defined_run_id

UUID('34bb3b7b-d0ff-4a39-99a2-212affb13732')

In [7]:
from langsmith import traceable

@traceable
def foo():
    return "This is a sample Run!"

We are passing in a config with our function call through `langsmith_extra` that contains our pre-defined run_id

In [8]:
foo(langsmith_extra={"run_id": pre_defined_run_id})

'This is a sample Run!'

Now we can directly create feedback on this run!

In [9]:
from langsmith import Client

client = Client()

client.create_feedback(pre_defined_run_id, "user_feedback", score=8)

C:\Users\reich\AppData\Local\Temp\ipykernel_15428\2980885594.py:5: LangSmithWarning: Creating feedback for a run without session_id is deprecated and will stop working in a future release. See https://docs.langchain.com/langsmith/smithdb-sdk-migration-feedback#feedback-create
  client.create_feedback(pre_defined_run_id, "user_feedback", score=8)


Feedback(id=UUID('01a136aa-d09c-7ab2-a26f-8dae8ec21247'), created_at=datetime.datetime(2026, 10, 13, 19, 8, 46, 620268, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2026, 10, 13, 19, 8, 46, 620272, tzinfo=datetime.timezone.utc), run_id=UUID('34bb3b7b-d0ff-4a39-99a2-212affb13732'), trace_id=None, key='user_feedback', score=8, value=None, comment=None, correction=None, feedback_source=FeedbackSourceBase(type='api', metadata={}, user_id=None, user_name=None), session_id=None, start_time=None, comparative_experiment_id=None, feedback_group_id=None, extra=None)

### Pre-signed Feedback URLs

![presigned url](../../images/presigned_url.png)


This can also be helpful for pre-signed feedback URLs. You would want to use these when you can't expose API keys or other secrets to the client, e.g. in a web application. Using a pre-determined run_id LangSmith has an endpoint create_presigned_feedback_token which will create a URL for sending feedback, without the use of secrets required.

In [10]:
pre_signed_url_id = uuid.uuid4()
pre_signed_url_id

UUID('59b3cbfa-239e-486a-9017-8bb06e08b737')

In [11]:
pre_signed_url = client.create_presigned_feedback_token(pre_signed_url_id, "user_presigned_feedback")

print(pre_signed_url)

id=UUID('dbb151f8-df00-4add-8540-f6759c5360bf') url='https://api.smith.langchain.com/feedback/tokens/dbb151f8-df00-4add-8540-f6759c5360bf' expires_at=datetime.datetime(2026, 10, 13, 22, 13, 49, 766952, tzinfo=TzInfo(0))


Here, we can see that even though we haven't created a run yet, we're still able to generate the feedback URL.

Now, let's invoke our chain so the run with that ID is created:

In [12]:
foo(langsmith_extra={"run_id": pre_signed_url_id})

'This is a sample Run!'

Then, once our run is created, we can use the feedback URL to send feedback:

In [13]:
import requests

url_with_score = f"{pre_signed_url.url}?score=1"

response = requests.get(url_with_score)

if response.status_code >= 200 and response.status_code < 300:
    print("Feedback submitted successfully!")
else:
    print("Feedback submission failed!")

Feedback submitted successfully!
